# Sentinel-2 Burn Severity Analysis

**Case study:** Wildfire in southern Europe, July–August 2021
**Data:** Sentinel-2 L2A (bands B04, B08, B12)
**Pre-fire:** 2021-07-28 | **Post-fire:** 2021-08-27

This notebook calculates the **Normalized Burn Ratio (NBR)**, derives **dNBR**,
classifies **burn severity** according to USGS thresholds, and performs spatial
analysis including vectorization and NDVI change detection.

## STEP 1 — Setup and imports

In [6]:
import sys
import os
import numpy as np
import rasterio
import rasterio.features
import geopandas as gpd
from pathlib import Path
from shapely.geometry import shape
from rasterio.transform import Affine
from PIL import Image, ImageDraw

# Project paths (MUSI być przed importem src/)
PROJECT_DIR = Path(r"C:\Users\Bronislaw\Documents\GitHub\sentinel2-burn-severity")
DATA = PROJECT_DIR / "data"
OUTPUTS = PROJECT_DIR / "outputs"
OUTPUTS.mkdir(exist_ok=True)

# Dodaj katalog projektu do sys.path — dopiero teraz Python widzi src/
sys.path.insert(0, str(PROJECT_DIR))

from src.indices import nbr, nbr2, dnbr, ndvi, classify_severity, SEVERITY_LABELS
from src.visualize import severity_to_rgb, array_to_rgb, save_png, SEVERITY_COLORS

print("✅ Libraries loaded")
print(f"   rasterio: {rasterio.__version__}")
print(f"   Project : {PROJECT_DIR}")

✅ Libraries loaded
   rasterio: 1.4.4
   Project : C:\Users\Bronislaw\Documents\GitHub\sentinel2-burn-severity


## STEP 2 — Define input band paths

Sentinel-2 L2A bands used:
- **B04** (Red, 665 nm) — for NDVI
- **B08** (NIR, 842 nm) — for NBR and NDVI
- **B11** (SWIR1, 1610 nm) — for NBR2
- **B12** (SWIR2, 2190 nm) — for NBR and NBR2

In [ ]:
# Pre-fire bands (2021-07-28)
PRE = DATA / "pre_fire"
pre_b04 = PRE / "2021-07-28-00_00_2021-07-28-23_59_Sentinel-2_L2A_B04_(Raw).tiff"
pre_b08 = PRE / "2021-07-28-00_00_2021-07-28-23_59_Sentinel-2_L2A_B08_(Raw).tiff"
pre_b11 = PRE / "2021-07-28-00_00_2021-07-28-23_59_Sentinel-2_L2A_B11_(Raw).tiff"
pre_b12 = PRE / "2021-07-28-00_00_2021-07-28-23_59_Sentinel-2_L2A_B12_(Raw).tiff"

# Post-fire bands (2021-08-27)
POST = DATA / "post_fire"
post_b04 = POST / "2021-08-27-00_00_2021-08-27-23_59_Sentinel-2_L2A_B04_(Raw).tiff"
post_b08 = POST / "2021-08-27-00_00_2021-08-27-23_59_Sentinel-2_L2A_B08_(Raw).tiff"
post_b11 = POST / "2021-08-27-00_00_2021-08-27-23_59_Sentinel-2_L2A_B11_(Raw).tiff"
post_b12 = POST / "2021-08-27-00_00_2021-08-27-23_59_Sentinel-2_L2A_B12_(Raw).tiff"

# Quick sanity check
with rasterio.open(pre_b08) as src:
    print(f"CRS         : {src.crs}")
    print(f"Dimensions  : {src.width} x {src.height} pixels")
    print(f"Bands       : {src.count}")
    print(f"Dtype       : {src.dtypes[0]}")
    print(f"Bounding box: {src.bounds}")

> **Study area:** ~43.27°N, 6.47°E (UTM 32T) — southern Massif des Maures,
> west of Grimaud, Var, France.

CRS         : EPSG:32632
Dimensions  : 1185 x 1127 pixels
Bands       : 1
Dtype       : uint8
Bounding box: BoundingBox(left=292145.066415, bottom=4789505.402311, right=300788.136788, top=4797432.268458)


## STEP 3 — Load bands and compute dNBR

**Formulas:**
- $NBR = \dfrac{NIR - SWIR}{NIR + SWIR}$
- $dNBR = NBR_{pre} - NBR_{post}$

Higher dNBR → more severe burn.

In [8]:
def load_band(path):
    """Load a single-band GeoTIFF as float32 array + profile."""
    with rasterio.open(path) as src:
        return src.read(1).astype(np.float32), src.profile

# Load all bands
pre_nir,  profile = load_band(pre_b08)
pre_swir, _       = load_band(pre_b12)
post_nir, _       = load_band(post_b08)
post_swir, _      = load_band(post_b12)

# Shape sanity check (B08 is 10 m, B12 is natively 20 m)
assert pre_nir.shape == pre_swir.shape == post_nir.shape == post_swir.shape, \
    "Bands must share the same shape — resample first!"

# NBR and dNBR
nbr_pre  = (pre_nir  - pre_swir)  / (pre_nir  + pre_swir  + 1e-10)
nbr_post = (post_nir - post_swir) / (post_nir + post_swir + 1e-10)
dnbr     = nbr_pre - nbr_post

print(f"Image shape : {pre_nir.shape}")
print(f"dNBR range  : {dnbr.min():.3f} to {dnbr.max():.3f}")

Image shape : (1127, 1185)
dNBR range  : -0.800 to 1.343


## STEP 4 — Classify burn severity

USGS burn severity thresholds (dNBR):

| Class       | dNBR range    | Code |
|-------------|---------------|------|
| Unburned    | < 0.10        | 0    |
| Low         | 0.10 – 0.27   | 1    |
| Moderate    | 0.27 – 0.44   | 2    |
| High        | 0.44 – 0.66   | 3    |
| Very High   | ≥ 0.66        | 4    |

In [9]:
severity = np.zeros_like(dnbr, dtype=np.uint8)
severity[(dnbr >= 0.10) & (dnbr < 0.27)] = 1
severity[(dnbr >= 0.27) & (dnbr < 0.44)] = 2
severity[(dnbr >= 0.44) & (dnbr < 0.66)] = 3
severity[ dnbr >= 0.66]                  = 4

labels = ['Unburned', 'Low', 'Moderate', 'High', 'Very High']
unique, counts = np.unique(severity, return_counts=True)
print("Pixel counts per class:")
for u, c in zip(unique, counts):
    print(f"  {labels[u]:<10} → {c:>8,} pixels")

Pixel counts per class:
  Unburned   →  887,735 pixels
  Low        →   85,504 pixels
  Moderate   →   56,021 pixels
  High       →   65,010 pixels
  Very High  →  241,225 pixels


## STEP 5 — Export raster products (GeoTIFF)

Save three georeferenced raster products:
1. `dnbr.tif` — continuous dNBR values (float32)
2. `severity.tif` — classified severity (uint8)
3. `severity_rgb.tif` — color-coded RGB map (uint8, 3 bands)

In [10]:
# Color map used across all outputs
color_map = {
    0: (34, 139, 34),    # Unburned  — green
    1: (255, 255, 0),    # Low       — yellow
    2: (255, 165, 0),    # Moderate  — orange
    3: (255, 0, 0),      # High      — red
    4: (139, 0, 0),      # Very High — dark red
}

# 1) dNBR as float32
profile.update(dtype=rasterio.float32, count=1)
with rasterio.open(OUTPUTS / 'dnbr.tif', 'w', **profile) as dst:
    dst.write(dnbr.astype(np.float32), 1)

# 2) Severity as uint8
profile.update(dtype=rasterio.uint8, count=1)
with rasterio.open(OUTPUTS / 'severity.tif', 'w', **profile) as dst:
    dst.write(severity, 1)

# 3) Severity as RGB GeoTIFF
h, w = severity.shape
rgb = np.zeros((3, h, w), dtype=np.uint8)
for val, color in color_map.items():
    mask = severity == val
    for band_idx, c in enumerate(color):
        rgb[band_idx][mask] = c

profile.update(dtype=rasterio.uint8, count=3)
with rasterio.open(OUTPUTS / 'severity_rgb.tif', 'w', **profile) as dst:
    dst.write(rgb)

print("✅ Exported GeoTIFFs:")
for f in ['dnbr.tif', 'severity.tif', 'severity_rgb.tif']:
    print(f"   outputs/{f}")

✅ Exported GeoTIFFs:
   outputs/dnbr.tif
   outputs/severity.tif
   outputs/severity_rgb.tif


## STEP 6 — Spatial statistics

Each Sentinel-2 pixel = 10 × 10 m = 100 m².

In [11]:
PIXEL_AREA_M2 = 10 * 10
total_pixels = severity.size

print("=" * 55)
print("  BURN SEVERITY — AREA STATISTICS")
print("=" * 55)
print(f"{'Class':<12} {'Pixels':>10} {'ha':>10} {'km²':>8} {'%':>7}")
print("-" * 55)

severity_stats = {}
for val, label in enumerate(labels):
    idx = np.where(unique == val)[0]
    n = int(counts[idx[0]]) if len(idx) > 0 else 0
    area_ha  = n * PIXEL_AREA_M2 / 1e4
    area_km2 = n * PIXEL_AREA_M2 / 1e6
    pct      = n / total_pixels * 100
    severity_stats[label] = {'pixels': n, 'ha': area_ha, 'km2': area_km2}
    print(f"  {label:<10} {n:>10,} {area_ha:>10.1f} {area_km2:>8.2f} {pct:>6.1f}%")

burned_ha = sum(v['ha'] for k, v in severity_stats.items() if k != 'Unburned')
print("-" * 55)
print(f"  {'Total burned':<10} {'':>10} {burned_ha:>10.1f} ha")
print("=" * 55)

  BURN SEVERITY — AREA STATISTICS
Class            Pixels         ha      km²       %
-------------------------------------------------------
  Unburned      887,735     8877.4    88.77   66.5%
  Low            85,504      855.0     8.55    6.4%
  Moderate       56,021      560.2     5.60    4.2%
  High           65,010      650.1     6.50    4.9%
  Very High     241,225     2412.2    24.12   18.1%
-------------------------------------------------------
  Total burned                4477.6 ha


## STEP 7 — NDVI change detection (context layer)

NDVI quantifies vegetation vigor. The post-fire drop in NDVI (dNDVI) is an
independent indicator of vegetation loss and complements dNBR.

$NDVI = \dfrac{NIR - RED}{NIR + RED}$

In [12]:
pre_red,  _ = load_band(pre_b04)
post_red, _ = load_band(post_b04)

ndvi_pre  = (pre_nir  - pre_red)  / (pre_nir  + pre_red  + 1e-10)
ndvi_post = (post_nir - post_red) / (post_nir + post_red + 1e-10)
dndvi     = ndvi_pre - ndvi_post

print(f"NDVI pre-fire  : {ndvi_pre.min():.3f}  to {ndvi_pre.max():.3f}")
print(f"NDVI post-fire : {ndvi_post.min():.3f}  to {ndvi_post.max():.3f}")
print(f"dNDVI          : {dndvi.min():.3f}  to {dndvi.max():.3f}")

# Export dNDVI
profile.update(dtype=rasterio.float32, count=1)
with rasterio.open(OUTPUTS / 'dndvi.tif', 'w', **profile) as dst:
    dst.write(dndvi.astype(np.float32), 1)
print("✅ Exported outputs/dndvi.tif")

NDVI pre-fire  : -1.000  to 0.977
NDVI post-fire : -1.000  to 0.957
dNDVI          : -1.103  to 0.836
✅ Exported outputs/dndvi.tif


## STEP 8 — NBR2 change detection (alternative burn index)

NBR2 uses both SWIR bands (B11 at 1610 nm and B12 at 2190 nm).
It is more sensitive to **soil moisture changes** and **char residue**
than standard NBR, making it a valuable complementary indicator.

$NBR2 = \dfrac{SWIR1 - SWIR2}{SWIR1 + SWIR2}$

In [13]:
# Load SWIR1 bands (B11)

pre_swir1, _  = load_band(pre_b11)
post_swir1, _ = load_band(post_b11)

# Sanity check — B11 and B12 must match
assert pre_swir1.shape == pre_swir.shape, \
    f"B11 shape {pre_swir1.shape} != B12 shape {pre_swir.shape}. Resample needed!"

# Compute NBR2 and dNBR2
nbr2_pre  = nbr2(pre_swir1, pre_swir)
nbr2_post = nbr2(post_swir1, post_swir)
dnbr2     = nbr2_pre - nbr2_post

print(f"NBR2 pre-fire  : {nbr2_pre.min():.3f}  to {nbr2_pre.max():.3f}")
print(f"NBR2 post-fire : {nbr2_post.min():.3f}  to {nbr2_post.max():.3f}")
print(f"dNBR2          : {dnbr2.min():.3f}  to {dnbr2.max():.3f}")

# Export dNBR2
profile.update(dtype=rasterio.float32, count=1)
with rasterio.open(OUTPUTS / 'dnbr2.tif', 'w', **profile) as dst:
    dst.write(dnbr2.astype(np.float32), 1)
print("✅ Exported outputs/dnbr2.tif")

NBR2 pre-fire  : -0.031  to 0.508
NBR2 post-fire : -0.200  to 0.476
dNBR2          : -0.333  to 0.571
✅ Exported outputs/dnbr2.tif


## STEP 9 — Vectorize burn perimeters (GeoJSON)

Convert the raster severity map into vector polygons for use in GIS software
(QGIS, ArcGIS, web maps). A downsampling factor keeps memory usage safe.

In [16]:
FACTOR = 5  # downsample factor (reduces memory ~25×)

sev_small = severity[::FACTOR, ::FACTOR].astype(np.int32)
new_transform = profile['transform'] * Affine.scale(FACTOR, FACTOR)
print(f"Original: {severity.shape} → Vectorized at: {sev_small.shape}")

shapes_gen = rasterio.features.shapes(
    sev_small,
    mask=sev_small > 0,          # burned areas only
    transform=new_transform,
)

records = [
    {
        'geometry': shape(geom),
        'properties': {'severity': int(value), 'label': labels[int(value)]},
    }
    for geom, value in shapes_gen
]

gdf = gpd.GeoDataFrame.from_records(records)
gdf = gdf.set_crs(profile['crs'])
gdf = gdf.to_crs(epsg=4326)  
gdf['area_ha'] = gdf.geometry.area / 1e4 
gdf['geometry'] = gdf.simplify(tolerance=10 * FACTOR, preserve_topology=True)

gdf.to_file(OUTPUTS / 'burn_severity.geojson', driver='GeoJSON')
print(f"✅ Vectorized → {len(gdf)} polygons")
print(f"   Total burned area (vector): {gdf['area_ha'].sum():.1f} ha")
print(f"   Saved to outputs/burn_severity.geojson")

Original: (1127, 1185) → Vectorized at: (226, 237)


C:\Users\Bronislaw\AppData\Local\Temp\ipykernel_17680\150348609.py:24: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['area_ha'] = gdf.geometry.area / 1e4


✅ Vectorized → 3715 polygons
   Total burned area (vector): 0.0 ha
   Saved to outputs/burn_severity.geojson


## STEP 10 — Visualization (PIL)

Composite PNG showing dNBR, burn severity classes, and dNDVI side by side.
PIL is used instead of matplotlib for better cross-platform stability.

In [15]:
# ── STEP 10 — Visualization (PIL, 2×2 grid) ──

from PIL import Image, ImageDraw

def array_to_rgb(arr, vmin, vmax, mode='fire'):
    """Simple colormap without matplotlib."""
    norm = np.clip((arr - vmin) / (vmax - vmin + 1e-10), 0, 1)
    if mode == 'ndvi':
        norm = 1 - norm
    h, w = norm.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    rgb[..., 0] = (norm * 255).astype(np.uint8)
    rgb[..., 1] = ((1 - np.abs(norm - 0.5) * 2) * 255).astype(np.uint8)
    return rgb

# Build 4 panels
img_dnbr  = array_to_rgb(dnbr,  vmin=-0.5, vmax=1.0, mode='fire')
img_dnbr2 = array_to_rgb(dnbr2, vmin=-0.3, vmax=0.5, mode='fire')
img_ndvi  = array_to_rgb(dndvi, vmin=-0.5, vmax=0.8, mode='ndvi')

h, w = severity.shape
img_sev = np.zeros((h, w, 3), dtype=np.uint8)
for val, color in color_map.items():
    img_sev[severity == val] = color

# Compose 2×2 grid
GAP = 10
PADDING = 40
total_w = w * 2 + GAP
total_h = h * 2 + GAP + PADDING
canvas  = np.full((total_h, total_w, 3), 255, dtype=np.uint8)

grid = [
    (img_dnbr,  'dNBR (B08-B12)',     0, 0),
    (img_sev,   'Burn Severity',       1, 0),
    (img_ndvi,  'dNDVI (B08-B04)',     0, 1),
    (img_dnbr2, 'dNBR2 (B11-B12)',     1, 1),
]

for panel, title, col, row in grid:
    x_off = col * (w + GAP)
    y_off = PADDING + row * (h + GAP)
    canvas[y_off:y_off + h, x_off:x_off + w] = panel

result = Image.fromarray(canvas)
draw = ImageDraw.Draw(result)
for _, title, col, row in grid:
    x_off = col * (w + GAP) + w // 2 - 50
    y_off = PADDING + row * (h + GAP) - 25
    draw.text((x_off, max(y_off, 5)), title, fill=(0, 0, 0))

result.save(OUTPUTS / 'overview.png')
print(f"✅ Saved outputs/overview.png ({total_w}×{total_h} px)")

# Standalone maps
Image.fromarray(img_sev).save(OUTPUTS / 'burn_severity.png')
print(f"✅ Saved outputs/burn_severity.png")

✅ Saved outputs/overview.png (2380×2304 px)
✅ Saved outputs/burn_severity.png


## Summary

**Output files in `outputs/`:**

| File | Format | Description |
|------|--------|-------------|
| `dnbr.tif` | GeoTIFF (float32) | Continuous dNBR values |
| `severity.tif` | GeoTIFF (uint8) | Classified severity (0–4) |
| `severity_rgb.tif` | GeoTIFF (uint8, RGB) | Color-coded severity map |
| `dndvi.tif` | GeoTIFF (float32) | NDVI change (vegetation loss) |
| `burn_severity.geojson` | GeoJSON | Vector polygons of burned areas |
| `burn_severity.png` | PNG | Standalone severity map |
| `overview.png` | PNG | dNBR + Severity + dNDVI composite |

**Next steps:**
- Load `burn_severity.geojson` into QGIS for interactive exploration
- Compare results with EFFIS/CEMS official burn perimeters for validation
- Build an interactive web map (folium / leaflet)